# Firm Characteristics Dataset

Academic dataset of anonymized firm characteristics for ML-based asset pricing.

| Property | Value |
|----------|-------|
| **Provider** | GitHub (Chen, Pelger, Zhu 2020) |
| **Asset Class** | US Equities (anonymized) |
| **Frequency** | Monthly |
| **Firms** | Anonymized |
| **Coverage** | 1967-2016 |
| **Size** | ~258 MB |
| **API Key** | None (free) |
| **Loader** | `load_firm_characteristics()` |

**NOTE**: This is a **static academic dataset**. Firms are anonymized and not updateable.

In [1]:
"""Firm Characteristics - download, explore, and update workflow."""

import json
import os
import sys
import zipfile
from pathlib import Path

import polars as pl

## 1. Configuration

This is a **static academic dataset** from Chen, Pelger, and Zhu (2020)
"Deep Learning in Asset Pricing". No local configuration file.

### Dataset Characteristics

- **94 firm characteristics**: Accounting ratios, technical indicators, etc.
- **Anonymized firms**: No stock identifiers to prevent data mining
- **Pre-split periods**: Train (1967-1989), Test (2000-2016)
- **Gap period**: 1990-1999 excluded to prevent look-ahead bias

In [2]:
print("=== Firm Characteristics Configuration ===")
print("Provider: GitHub (Chen, Pelger, Zhu 2020)")
print("Paper: 'Deep Learning in Asset Pricing'")
print("Coverage: 1967-1989 (train), 2000-2016 (test)")
print("Features: 94 firm characteristics")
print("Frequency: Monthly")
print("\nThis is a static academic dataset. Firms are anonymized.")

=== Firm Characteristics Configuration ===
Provider: GitHub (Chen, Pelger, Zhu 2020)
Paper: 'Deep Learning in Asset Pricing'
Coverage: 1967-1989 (train), 2000-2016 (test)
Features: 94 firm characteristics
Frequency: Monthly

This is a static academic dataset. Firms are anonymized.


## 2. API Key Setup

**No API key required.** This dataset is freely available on GitHub.

In [3]:
print("No API key required - data is hosted on GitHub.")
print("Source: https://github.com/jasonzy121/Deep_Learning_Asset_Pricing")

No API key required - data is hosted on GitHub.
Source: https://github.com/jasonzy121/Deep_Learning_Asset_Pricing


## 3. Download Data

The dataset is hosted on Google Drive via the GitHub repository.
Download can be automatic (with `gdown`) or manual.

In [4]:
# Google Drive file ID for the data archive
GDRIVE_FILE_ID = "1nYHpJ2lNm-qDX5iq18-HaL1H6z7lPGVi"
GITHUB_REPO = "https://github.com/jasonzy121/Deep_Learning_Asset_Pricing"


def download_firm_characteristics(dry_run: bool = False, force: bool = False, convert: bool = True):
    """Download Chen-Pelger-Zhu firm characteristics dataset.

    Args:
        dry_run: If True, show what would be downloaded without doing it
        force: If True, re-download even if data exists
        convert: If True, convert CSV to parquet after download
    """
    from utils import ML4T_DATA_PATH

    output_dir = ML4T_DATA_PATH / "academic"
    dl_dir = output_dir / "dl_asset_pricing"
    parquet_path = output_dir / "firm_characteristics_all.parquet"

    print("=== Firm Characteristics Download ===")
    print("Dataset: Chen-Pelger-Zhu (2020)")
    print("Coverage: 1967-1989 (train), 2000-2016 (test)")
    print("Features: 94 firm characteristics + returns")
    print("Estimated size: ~1.5 GB (archive), ~258 MB (parquet)")
    print(f"Output: {output_dir}")

    if dry_run:
        print("\n[DRY RUN] Would download:")
        print("  - data.zip from Google Drive (~1.5 GB)")
        print(f"  - Extract to: {dl_dir}")
        print(f"  - Convert to parquet: {parquet_path}")
        print("\nManual download:")
        print(f"  1. Go to: {GITHUB_REPO}")
        print("  2. Download data.zip from Google Drive link")
        print(f"  3. Extract to: {dl_dir}")
        return

    # Check existing
    if parquet_path.exists() and not force:
        existing = pl.read_parquet(parquet_path)
        print(f"\nData already exists ({len(existing):,} rows).")
        print("Use force=True to re-download.")
        return

    # Try automatic download with gdown
    try:
        import gdown
    except ImportError:
        print("\nWARNING: gdown not installed for automatic download")
        print("Install with: pip install gdown")
        print("\nManual download instructions:")
        print(f"  1. Go to: {GITHUB_REPO}")
        print("  2. Download data.zip from Google Drive link")
        print(f"  3. Extract to: {dl_dir}")
        print("  4. Run: download_firm_characteristics(convert=True)")
        return

    dl_dir.mkdir(parents=True, exist_ok=True)
    zip_path = dl_dir / "data.zip"

    print("\nDownloading from Google Drive...")
    url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"
    gdown.download(url, str(zip_path), quiet=False)

    if not zip_path.exists():
        print("ERROR: Download failed")
        return

    print("\nExtracting archive...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dl_dir)

    # Cleanup zip
    zip_path.unlink()

    print("Download complete!")

    # Convert to parquet
    if convert:
        _convert_to_parquet(output_dir)


def _convert_to_parquet(output_dir: Path):
    """Convert RetChar.csv to parquet format with train/test splits."""
    dl_dir = output_dir / "dl_asset_pricing"
    retchar_path = dl_dir / "RetChar.csv"

    if not retchar_path.exists():
        # Try nested path
        nested_path = dl_dir / "data" / "RetChar.csv"
        if nested_path.exists():
            retchar_path = nested_path
        else:
            print(f"ERROR: RetChar.csv not found at {retchar_path}")
            return

    print("\nConverting to parquet format...")

    # Read CSV
    df = pl.read_csv(retchar_path)
    print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")

    # Date is in YYYYMMDD format (integer)
    df = df.with_columns(
        pl.col("Date").cast(pl.Utf8).str.to_date("%Y%m%d").alias("date"),
        pl.col("Date").cast(pl.Utf8).str.slice(0, 4).cast(pl.Int32).alias("year"),
    )

    # Rename permno column if it exists differently
    if "Permno" in df.columns:
        df = df.rename({"Permno": "permno"})
    if "RET" in df.columns:
        df = df.rename({"RET": "ret"})

    # Create splits based on paper
    train_df = df.filter(pl.col("year") < 1990)
    test_df = df.filter(pl.col("year") >= 2000)

    # Drop helper columns
    train_df = train_df.drop(["Date", "year"])
    test_df = test_df.drop(["Date", "year"])
    all_df = df.drop(["Date", "year"])

    # Save parquet files
    output_dir.mkdir(parents=True, exist_ok=True)

    all_df.write_parquet(output_dir / "firm_characteristics_all.parquet")
    train_df.write_parquet(output_dir / "firm_characteristics_train.parquet")
    test_df.write_parquet(output_dir / "firm_characteristics_test.parquet")

    print("  Created:")
    print(f"    firm_characteristics_all.parquet: {len(all_df):,} rows")
    print(f"    firm_characteristics_train.parquet: {len(train_df):,} rows (1967-1989)")
    print(f"    firm_characteristics_test.parquet: {len(test_df):,} rows (2000-2016)")

### Download

In [5]:
# Uncomment to download
# download_firm_characteristics()

### Dry Run (Preview)

In [6]:
download_firm_characteristics(dry_run=True)

=== Firm Characteristics Download ===
Dataset: Chen-Pelger-Zhu (2020)
Coverage: 1967-1989 (train), 2000-2016 (test)
Features: 94 firm characteristics + returns
Estimated size: ~1.5 GB (archive), ~258 MB (parquet)
Output: data/academic

[DRY RUN] Would download:
  - data.zip from Google Drive (~1.5 GB)
  - Extract to: data/academic/dl_asset_pricing
  - Convert to parquet: data/academic/firm_characteristics_all.parquet

Manual download:
  1. Go to: https://github.com/jasonzy121/Deep_Learning_Asset_Pricing
  2. Download data.zip from Google Drive link
  3. Extract to: data/academic/dl_asset_pricing


## 4. Load and Explore

Once downloaded, use the loader throughout the book:

In [7]:
from data import load_firm_characteristics

# Load the full dataset
df = load_firm_characteristics()

print(f"Shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# Count features (exclude timestamp, symbol, ret)
feature_cols = [c for c in df.columns if c not in ["timestamp", "symbol", "ret"]]
print(f"Features: {len(feature_cols)}")
print(f"Memory: {df.estimated_size('mb'):.1f} MB")

Shape: (1218555, 49)
Date range: 1967-01-31 to 2016-12-30
Features: 47
Memory: 446.9 MB


In [8]:
# Schema
df.schema

Schema([('ret', Float64),
        ('A2ME', Float64),
        ('AC', Float64),
        ('AT', Float64),
        ('ATO', Float64),
        ('BEME', Float64),
        ('Beta', Float64),
        ('C', Float64),
        ('CF', Float64),
        ('CF2P', Float64),
        ('CTO', Float64),
        ('D2A', Float64),
        ('D2P', Float64),
        ('DPI2A', Float64),
        ('E2P', Float64),
        ('FC2Y', Float64),
        ('IdioVol', Float64),
        ('Investment', Float64),
        ('Lev', Float64),
        ('LME', Float64),
        ('LT_Rev', Float64),
        ('LTurnover', Float64),
        ('MktBeta', Float64),
        ('NI', Float64),
        ('NOA', Float64),
        ('OA', Float64),
        ('OL', Float64),
        ('OP', Float64),
        ('PCM', Float64),
        ('PM', Float64),
        ('PROF', Float64),
        ('Q', Float64),
        ('r2_1', Float64),
        ('r12_2', Float64),
        ('r12_7', Float64),
        ('r36_13', Float64),
        ('Rel2High', Float64),
     

In [9]:
# Preview
df.head(10)

ret,A2ME,AC,AT,ATO,BEME,Beta,C,CF,CF2P,CTO,D2A,D2P,DPI2A,E2P,FC2Y,IdioVol,Investment,Lev,LME,LT_Rev,LTurnover,MktBeta,NI,NOA,OA,OL,OP,PCM,PM,PROF,Q,r2_1,r12_2,r12_7,r36_13,Rel2High,Resid_Var,RNA,ROA,ROE,S2P,SGA2S,Spread,ST_REV,SUV,Variance,timestamp,split
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,str
0.136223,0.108392,0.24359,0.152681,-0.182984,0.113054,0.185315,0.113054,-0.271562,0.404429,-0.252914,0.036131,0.15035,0.001166,0.343823,-0.381119,-0.155012,0.073427,-0.01049,0.01049,0.432401,0.059441,-0.136364,-0.390443,0.134033,0.259907,-0.245921,0.143357,-0.148019,0.311189,-0.283217,-0.217949,-0.008159,-0.043124,0.250583,0.106061,-0.122378,-0.224942,0.045455,0.059441,0.08042,-0.015152,-0.38345,-0.164336,-0.241259,-0.325175,-0.182984,1967-01-31,"""train"""
0.317129,0.455711,-0.36014,-0.472028,0.257576,0.45338,0.318182,0.122378,0.071096,0.43007,0.180653,-0.194639,-0.5,-0.488345,0.488345,-0.201632,0.495338,-0.467366,0.17366,-0.488345,-0.439394,0.332168,-0.145688,-0.320513,-0.294872,-0.334499,0.355478,-0.357809,-0.409091,-0.392774,-0.068765,0.015152,0.082751,0.364802,0.495338,-0.329837,-0.490676,0.483683,0.038462,-0.346154,-0.245921,0.439394,-0.175991,0.439394,-0.399767,-0.064103,0.483683,1967-01-31,"""train"""
0.101064,0.050117,0.050117,0.206294,-0.276224,0.096737,0.068765,-0.294872,-0.325175,0.374126,-0.325175,0.099068,0.148019,-0.115385,0.019814,-0.029138,-0.369464,-0.182984,0.003497,0.10373,-0.05711,0.138695,-0.085082,0.175991,0.269231,0.047786,-0.280886,-0.110723,0.124709,0.203963,-0.194639,-0.187646,-0.334499,-0.294872,-0.320513,-0.047786,0.206294,-0.129371,-0.145688,-0.136364,-0.134033,-0.108392,0.001166,0.115385,0.5,0.273893,0.026807,1967-01-31,"""train"""
0.287367,-0.031469,0.019814,0.416084,-0.252914,0.005828,-0.297203,-0.493007,-0.31352,0.315851,-0.320513,0.416084,-0.208625,0.213287,-0.054779,-0.054779,-0.413753,0.024476,-0.078089,0.339161,-0.297203,0.15035,-0.33683,-0.166667,0.236597,0.001166,-0.31352,0.040793,0.092075,0.15035,-0.236597,-0.108392,-0.399767,-0.341492,-0.378788,-0.423077,-0.271562,-0.460373,-0.136364,-0.059441,-0.099068,-0.17366,-0.178322,0.024476,-0.001166,0.378788,-0.357809,1967-01-31,"""train"""
0.143427,0.371795,0.497669,0.280886,-0.175991,0.362471,0.325175,-0.346154,-0.08042,0.148019,-0.141026,-0.129371,-0.168998,-0.297203,0.120047,-0.213287,0.110723,-0.085082,-0.120047,-0.003497,0.180653,0.388112,0.378788,-0.318182,0.24359,0.483683,-0.038462,-0.320513,-0.332168,-0.304196,-0.238928,-0.385781,-0.38345,-0.427739,-0.162005,0.374126,-0.416084,0.071096,-0.278555,-0.392774,-0.38345,0.299534,-0.189977,0.292541,0.152681,-0.304196,0.248252,1967-01-31,"""train"""
0.162367,-0.071096,-0.472028,0.283217,0.012821,-0.134033,0.208625,0.483683,-0.248252,0.050117,-0.304196,-0.203963,-0.164336,0.234266,0.31352,-0.2669,0.192308,0.399767,-0.071096,0.259907,-0.026807,-0.075758,0.113054,0.178322,-0.106061,-0.467366,-0.332168,-0.285548,-0.185315,0.043124,-0.374126,-0.01049,-0.229604,-0.017483,0.168998,-0.178322,-0.250583,0.259907,0.061772,0.36014,0.378788,-0.276224,-0.255245,0.229604,0.2669,-0.001166,0.224942,1967-01-31,"""train"""
0.157643,-0.089744,-0.45338,0.350816,-0.308858,0.026807,-0.378788,-0.469697,0.285548,-0.224942,-0.367133,-0.497669,-0.187646,-0.339161,0.24359,-0.001166,-0.131702,-0.36014,-0.192308,0.327506,-0.357809,-0.259907,-0.019814,-0.388112,0.378788,-0.465035,-0.367133,-0.038462,0.2669,0.462704,-0.234266,-0.085082,-0.343823,-0.08042,-0.192308,-0.003497,0.0338,-0.236597,0.008159,0.194639,0.026807,-0.252914,0.0338,-0.17366,-0.10373,-0.138695,-0.283217,1967-01-31,"""train"""
-0.081223,-0.141026,0.129371,0.061772,0.148019,-0.241259,0.099068,0.397436,0.019814,-0.353147,0.299534,-0.250583,-0.012821,-0.315851,-0.287879,-0.5,0.369464,0.215618,-0.036131,0.138695,0.203963,0.411422,0.299534,0.199301,-0.05711,0.092075,0.332168,0.038462,-0.493007,-0.252914,-0.423077,0.078

### Feature Overview

In [10]:
# Feature statistics
feature_cols = [c for c in df.columns if c not in ["timestamp", "symbol", "split", "ret"]]
print(f"Number of features: {len(feature_cols)}")
print("\nFeature names (first 20):")
for col in feature_cols[:20]:
    print(f"  {col}")
if len(feature_cols) > 20:
    print(f"  ... and {len(feature_cols) - 20} more")

Number of features: 46

Feature names (first 20):
  A2ME
  AC
  AT
  ATO
  BEME
  Beta
  C
  CF
  CF2P
  CTO
  D2A
  D2P
  DPI2A
  E2P
  FC2Y
  IdioVol
  Investment
  Lev
  LME
  LT_Rev
  ... and 26 more


### Train/Test Split Coverage

In [11]:
# Yearly coverage
yearly = (
    df.with_columns(pl.col("timestamp").dt.year().alias("year"))
    .group_by("year")
    .agg(
        pl.len().alias("n_observations"),
    )
    .sort("year")
)
print("Yearly coverage:")
yearly

Yearly coverage:


year,n_observations
i32,u32
1967,7009
1968,10192
1969,11767
1970,12634
1971,13570
…,…
2012,27039
2013,26533
2014,25438


## 5. Data Profile

In [12]:
from utils import ML4T_DATA_PATH

# Check for existing profile
profile_path = ML4T_DATA_PATH / "academic" / "firm_characteristics_profile.json"

if profile_path.exists():
    profile = json.loads(profile_path.read_text())
    print("=== Firm Characteristics Profile ===")
    print(f"Rows: {profile['rows']:,}")
    print(f"Columns: {profile['columns']}")
    print(f"Memory: {profile['memory_mb']:.1f} MB")
else:
    print(f"Profile not found at {profile_path}")
    print("Generate with: python generate_profiles.py --dataset firm_characteristics")

Profile not found at data/academic/firm_characteristics_profile.json
Generate with: python generate_profiles.py --dataset firm_characteristics


## 6. Loader Options

The loader supports loading the full dataset or pre-defined splits:

In [13]:
# Load train split (1967-1989)
train = load_firm_characteristics(split="train")
print(f"Train: {train.shape}, {train['timestamp'].min()} to {train['timestamp'].max()}")

Train: (414025, 49), 1967-01-31 to 1989-12-29


In [14]:
# Load test split (2000-2016)
test = load_firm_characteristics(split="test")
print(f"Test: {test.shape}, {test['timestamp'].min()} to {test['timestamp'].max()}")

Test: (497436, 49), 2000-01-31 to 2016-12-30


In [15]:
# Load full dataset (default)
full = load_firm_characteristics()
print(f"Full: {full.shape}")

Full: (1218555, 49)


## 7. Documentation

### Source

- **Paper**: Chen, Pelger, Zhu (2020) "Deep Learning in Asset Pricing"
- **GitHub**: https://github.com/jasonzy121/Deep_Learning_Asset_Pricing
- **Published**: Management Science, 2024

### Dataset Columns

| Column | Description |
|--------|-------------|
| `date` | Month-end date |
| `permno` | Anonymized firm identifier |
| `ret` | Monthly stock return |
| `me` | Market equity |
| `bm` | Book-to-market ratio |
| `mom12m` | 12-month momentum |
| `... (94 total)` | Various firm characteristics |

### Train/Test Split

| Split | Period | Purpose |
|-------|--------|---------|
| Train | 1967-1989 | Model training |
| Gap | 1990-1999 | Excluded (prevents look-ahead) |
| Test | 2000-2016 | Out-of-sample evaluation |

### Data Quality Notes

- **Anonymized firms**: No stock identifiers to prevent data mining
- **Cross-sectional ranking**: Features are rank-transformed
- **Missing values**: Some characteristics have gaps
- **Survivorship**: Includes delisted firms

## 8. Updating Data

**This dataset is NOT updateable.**

This is a static academic dataset published with a research paper.
The data cannot be extended beyond the original publication period (2016).

### Related Resources

For more recent firm characteristics data, consider:

| Resource | Coverage | Access |
|----------|----------|--------|
| WRDS/CRSP | 1926-present | Subscription |
| Open Source Asset Pricing | Varies | Free |
| Ken French Library | 1926-present | Free (factors only) |

## Summary

| Item | Value |
|------|-------|
| Features | 94 firm characteristics |
| Frequency | Monthly |
| Coverage | 1967-2016 (with gap 1990-1999) |
| Provider | GitHub (Chen, Pelger, Zhu 2020) |
| Loader | `load_firm_characteristics(split=None)` |

**Primary use**: Deep learning asset pricing research (Chapters 10-15).
**Limitation**: Anonymized firms, static dataset ends 2016.